# 03 - Revenue Analysis (Phân tích Doanh thu)

**Mục đích:** 
Giải quyết nhóm câu hỏi nghiên cứu **Q1**: *"Hoạt động bán hàng thay đổi như thế nào theo thời gian, sản phẩm, và quốc gia?"*

**Nguyên tắc quan trọng:**
1. **Revenue ≠ Profit:** Tuyệt đối không gọi Doanh thu là Lợi nhuận vì dataset không có chi phí vốn hàng bán (COGS).
2. **Phân biệt rõ:** 
   - `Revenue`: Doanh thu ở cấp độ từng dòng giao dịch mặt hàng.
   - `InvoiceValue`: Tổng giá trị của một hóa đơn (`SUM(Revenue)`).
3. **Tôn trọng tính khách quan của dữ liệu:** Không suy diễn nguyên nhân tiếp thị / chiến dịch khi dữ liệu không ghi nhận.

## 0. Chuẩn bị thư viện & Nạp dữ liệu đã làm sạch

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Thiết lập đường dẫn gốc an toàn
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.analysis.revenue_analysis import (
    analyze_monthly_revenue,
    analyze_product_revenue,
    analyze_country_revenue,
    analyze_invoice_value_distribution,
)

pd.options.display.float_format = "{:.2f}".format

# Đọc dữ liệu đã qua làm sạch ở Phase 3
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "retail_cleaned.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
invoice_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "invoice_data.csv")

print(f"Dữ liệu giao dịch (retail_cleaned): {df.shape}")
print(f"Dữ liệu hóa đơn (invoice_data)     : {invoice_df.shape}")

## 1. Phân tích Doanh thu theo Thời gian (Monthly Revenue)

**Câu hỏi:** Doanh thu biến động ra sao qua từng tháng trong giai đoạn 12/2010 – 12/2011?

In [ ]:
monthly_rev = analyze_monthly_revenue(df)
monthly_rev

**Quan sát & Nhận định:**
- Doanh thu duy trì ổn định ở mức 500k – 700k bảng trong nửa đầu năm 2011.
- Bắt đầu tăng mạnh từ tháng 9/2011 (>1.01M bảng), đạt đỉnh vào tháng 11/2011 (>1.45M bảng). Đây là xu hướng mùa vụ rõ rệt (dịp lễ mua sắm cuối năm Black Friday / chuẩn bị Giáng sinh).
- **Lưu ý hạn chế của dữ liệu:** Tháng 12/2011 doanh thu thấp (432k bảng) vì **chỉ có dữ liệu đến ngày 09/12/2011 (thiếu 22 ngày cuối tháng)**, không phải do suy thoái doanh số.

## 2. Phân tích Doanh thu theo Sản phẩm (Product Revenue)

**Câu hỏi:** Những sản phẩm nào đóng góp doanh thu lớn nhất cho doanh nghiệp?

In [ ]:
top_products = analyze_product_revenue(df, top_n=10, exclude_special_codes=True)
top_products

**Quan sát & Nhận định:**
- Mặt hàng dẫn đầu doanh thu là `22423` - `REGENCY CAKESTAND 3 TIER` với hơn £174,000.
- Mặt hàng `23843` - `PAPER CRAFT , LITTLE BIRDIE` có số lượng cực lớn (80.995 chiếc) đóng góp £168,469 (đây là một đơn mua sỉ lớn).
- Các sản phẩm chủ lực thuộc nhóm trang trí gia đình, tiệc tùng và quà lưu niệm.

## 3. Phân tích Doanh thu theo Quốc gia (Country Revenue)

**Câu hỏi:** Thị trường địa lý phân bố ra sao? Doanh nghiệp phụ thuộc vào thị trường nào?

In [ ]:
top_countries = analyze_country_revenue(df, top_n=10)
top_countries

**Quan sát & Nhận định:**
- **Vương quốc Anh (United Kingdom)** là thị trường tuyệt đối áp đảo, đóng góp **84,0%** tổng doanh thu (gần 8,2 triệu bảng).
- Thị trường quốc tế xếp sau gồm: Hà Lan (£284k, 2.9%), EIRE (£263k, 2.7%), Đức (£221k, 2.3%), Pháp (£197k, 2.0%).

## 4. Phân phối Giá trị Hóa đơn (Invoice Value Distribution)

**Câu hỏi:** Một đơn hàng trung bình trị giá bao nhiêu? Phân phối có bị lệch không?

In [ ]:
inv_stats = analyze_invoice_value_distribution(invoice_df)
inv_stats

**Quan sát & Nhận định:**
- **Giá trị trung bình (Mean):** £533.17
- **Trung vị (Median):** £303.30
- **Nhận xét thống kê:** Mean lớn hơn đáng kể so với Median (hệ số Skewness dương rất cao). Điều này chỉ ra phân phối giá trị hóa đơn **lệch phải cực mạnh (right-skewed)**, bị chi phối bởi các đơn hàng mua buôn (wholesale) có giá trị hàng chục nghìn bảng.